# Google Hash Code 2017: Streaming Videos Optimization

## 1. Definizione del Problema

Lo scenario proposto riguarda l'ottimizzazione della distribuzione di video in una rete di streaming composta da quattro entità principali.

#### Data Center
Il Data Center contiene **tutti** i video disponibili.
* È il punto di riferimento centrale e garantisce che ogni richiesta possa essere soddisfatta, anche se non ci sono cache disponibili.
* La latenza verso il Data Center ($L_D$) è generalmente alta (migliaia di chilometri di distanza).

#### Cache Servers
Sono server periferici con capacità limitata.
* Ogni cache ha una capacità massima fissa di $X$ Megabyte.
* Un video può essere replicato **su più cache, o su nessuna**.
* La latenza verso una cache ($L_C$) è significativamente inferiore rispetto al Data Center.

#### Endpoints
Un endpoint rappresenta un gruppo di utenti situati nella stessa area geografica (es. un quartiere)
* Ogni endpoint ha una connessione diretta al Data Center con una latenza specifica ($L_D$).
* Può essere connesso a **zero, uno o più** cache server, ognuno con una propria latenza ($L_C$).

#### Videos & Requests
* Ogni video ha una dimensione specifica in MB.
* Il carico di lavoro è definito dalle **Richieste**: un numero previsto di volte in cui uno specifico video viene richiesto da uno specifico endpoint.

<div align="center">
    <img src="immagini/topologia.png" width="50%">
</div>

### Recap

- $\sum_{v \in Cache_i} Size(v) \leq X$ per ogni cache
- Un video può essere presente in **zero, una o più cache** contemporaneamente (replicazione libera)
- Una cache può contenere **zero o più video** diversi (fino al limite di capacità)
- Ogni richiesta viene sempre servita dalla sorgente con latenza minima disponibile
- Se nessuna cache connessa ha il video, si usa il Data Center (sempre disponibile)

## 2. Obiettivo e Funzione di Scoring


L'obiettivo della competizione è **massimizzare il tempo totale risparmiato** servendo i video dalle cache piuttosto che dal Data Center.

Il punteggio finale corrisponde alla **media del tempo risparmiato per richiesta**, espressa in **microsecondi**.

Per ogni richiesta $(R_v, R_e, R_n)$ dove vengono richiesti $R_n$ video $R_v$ dall'endpoint $R_e$:
1.  Il sistema seleziona la sorgente con la latenza minore $L$:
    $$L = \min(L_{DC}, L_{Cache_1}, L_{Cache_2}, \dots)$$
    *Se il video non è in nessuna cache connessa, $L = L_{DC}$.*

2.  Il tempo risparmiato per singola richiesta è:
    $$Saved_{req} = L_{DC} - L$$

3.  Lo Score totale è la somma ponderata dei risparmi, normalizzata sul numero totale di richieste e moltiplicata per 1000 (per convertire da millisecondi a microsecondi):

$$Score = \left( \frac{\sum (L_{DC} - L_{best}) \cdot N_{requests}}{\sum N_{total\_requests}} \right) \times 1000$$

### Esempio di Calcolo
Immaginiamo uno scenario con due endpoint:
<div align="center">
    <img src="immagini/esempio.png" width="45%">
</div>

* **Endpoint 0** ($L_{DC} = 1000ms$):
    * Richiede 1500 volte il Video #3. Il video è nella Cache 1 (latenza 300ms).
        * Risparmio: $(1000 - 300) = 700ms$.
    * Richiede 500 volte il Video #4. Il video non è in cache.
        * Risparmio: $(1000 - 1000) = 0ms$.
    * Richiede 1000 volte il Video #1. Il video è nella Cache 2 (latenza 200ms).
        * Risparmio: $(1000 - 200) = 800ms$.

* **Endpoint 1**:
    * Richiede 1000 volte il Video #0. Il video non è in cache.
        * Risparmio: $0ms$.

**Calcolo Finale:**
$$\text{Avg Saved} = \frac{(1500 \cdot 700) + (500 \cdot 0) + (1000 \cdot 800) + (1000 \cdot 0)}{1500 + 500 + 1000 + 1000}$$

$$\text{Avg Saved} = \frac{1,050,000 + 800,000}{4000} = \frac{1,850,000}{4000} = 462.5 \ ms$$

**Score Finale:** $462.5 \times 1000 = \mathbf{462,500}$ punti.

## 3. Modellazione dati e Parsing

### Formato di Input
Il dataset è fornito come file di testo ASCII contenente:
* **V, E, R, C, X:** Numero di Video, Endpoint, Richieste, Cache e Capacità della Cache ($X$ MB).
* **Video Sizes:** Una riga con le dimensioni di ogni video.
* **Endpoints:** Per ogni endpoint, la latenza verso il Data Center ($L_D$) e la lista delle cache connesse con relative latenze ($L_C$).
* **Requests:** Tripletta $(Video ID, Endpoint ID, Numero Richieste)$.

### Esempio
<div align="center">
    <img src="immagini/esempio_input.png" width="70%">
</div>

### Problem dataclass e Strutture Dati

Per ridurre l'occupazione di memoria e minimizzare i tempi di accesso durante l'esecuzione degli algoritmi ($O(1)$  per i dizionari) si evita l'uso di matrici dense (NumPy standard o Pandas) a favore delle seguenti strutture dati.

| Nome | Mapping | Funzione |
| :--- | :--- | :--- |
| **requests_map** | (video, endpoint) -> N_richieste | Restituisce il numero di richieste per un video da un dato endpoint. |
| **cache_to_endpoints** | cache -> [endpoint_collegati] | Identifica rapidamente quali endpoint sono serviti da una specifica cache. |
| **cache_endpoint_latency** | (cache, endpoint) -> latenza | Fornisce la latenza di connessione cache-endpoint. |
| **cache_to_requested_videos** | cache -> [video_richiesti] | Contiene l'insieme dei video richiesti dagli endpoint connessi a una cache. |
| **endpoints** | endpoint_id -> { datacenter latency, [(cache id1, latency1), (cache id2, latency2),  . . . ] } | Memorizza la latenza base verso il Data Center ($L_{DC}$) e le latenze verso tutte le cache collegate. |


In [1]:
import numpy as np
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Dict, Set, Tuple, Iterator
from tqdm import tqdm

@dataclass # con dataclass non serve scrivere il metodo __init__
class ProblemData:
    # Dati di input
    V: int  # Numero di video
    E: int  # Numero di endpoint
    R: int  # Numero di richieste
    C: int  # Numero di cache
    X: int  # Capacità di ogni cache
    video_sizes: np.array # array contenente dimensione del video i-esimo

    # Lista di dizionari con informazioni sugli endpoint, strutturata così:
    # endpoint_id -> { 'datacenter_latency': int,
    #                  'cache_connections': [(cache_id, latency), ...]}
    endpoints: List[dict] #dict = dizionario generico in python. (Dict è un tipo specifico di dato, type hinting)
    requests_map: Dict[Tuple[int, int], int]            # (video, endpoint) -> n_requests
    cache_to_endpoints: Dict[int, List[int]]            # cache -> [endpoint_id1, ...]
    cache_endpoint_latency: Dict[Tuple[int, int], int]  # (cache, endpoint) -> latency
    cache_to_requested_videos: Dict[int, Set[int]]      # cache -> set di video id richiesti
    
    total_requests: int # Somma totale di tutte le richieste


### Caricamento Input

La funzione `load_data` si occupa del parsing completo del file di input, costruendo l'oggetto `ProblemData` con tutte le strutture dati necessarie. 

Dopo aver aperto il file e creato un iteratore sulle righe, estrae prima l'header contenente i parametri principali (V, E, R, C, X) e le dimensioni dei video

Il parsing vero e proprio è delegato a due funzioni helper. La funzione `_parse_endpoints()` si occupa di leggere le informazioni sugli endpoint, costruendo contemporaneamente le mappe `cache_to_endpoints` e `cache_endpoint_latency`. 

Successivamente, `_parse_requests()` processa tutte le richieste video-endpoint, popolando la mappa `requests_map` e identificando quali video sono richiesti dagli endpoint connessi a ciascuna cache.


In [2]:
from utils import _parse_endpoints, _parse_requests

INPUT_FILE = r"..\hashcode_2017_qualification_round.in\me_at_the_zoo.in"

def load_data(filename: str) -> ProblemData:

    print(f"Reading {filename}...")

    #la funzione with si occupa di chiudere automaticamente il file dopo l'indentazione
    with open(filename, 'r') as f:
        #la funzione iter() crea un oggetto iteratore che restituisce una riga alla volta quando viene chiamato next()
        #la funzione strip() rimuove gli spazi bianchi iniziali e finali di ogni riga
        lines_iter = iter(line.strip() for line in f)
        
        "HEADER E VIDEO SIZES"
        # split() spezza la stringa "100 10 1000 5 500" in una lista ["100", "10", ...], 
        # map(int, ...) li converte tutti da testo a numero intero
        V, E, R, C, X = map(int, next(lines_iter).split())
        total_steps_bar = E + R

        with tqdm(total=total_steps_bar, desc="Parsing Data", unit="step") as pbar:

            #idem ma trasformo in array numpy per ottimizzare calcoli 
            video_sizes = np.array(list(map(int, next(lines_iter).split())))
            
            "ENDPOINTS"
            endpoints, c_to_e, c_e_lat = _parse_endpoints(lines_iter, E, pbar)
            
            "REQUESTS"
            req_map, c_req_vids, tot_req = _parse_requests(lines_iter, R, endpoints, pbar)
    
    return ProblemData(
        V, E, R, C, X, video_sizes, endpoints,
        req_map, c_to_e, c_e_lat, c_req_vids,
        tot_req
    )

data = load_data(INPUT_FILE)
print(f"Data loaded successfully: {data.V} video, {data.E} endpoint, {data.R} requests, {data.C} caches, cache capacity {data.X}MB")
#print(f"Output file: {output_file}")

Reading ..\hashcode_2017_qualification_round.in\me_at_the_zoo.in...


Parsing Data: 100%|██████████| 110/110 [00:00<00:00, 57686.10step/s]

Data loaded successfully: 100 video, 10 endpoint, 100 requests, 10 caches, cache capacity 100MB


## 4. Algoritmo Greedy

Per costruire una soluzione iniziale valida, si implementano due varianti di un algoritmo Greedy. Entrambe le strategie mirano a riempire le cache con i video più "preziosi", utilizzando una **Priority Queue (Heap)** per selezionare iterativamente i candidati migliori.

Per massimizzare il riutilizzo del codice e garantire modularità, entrambi gli algoritmi si basano sulle stesse funzioni

### Calcolo beneficio: `compute_benefit`
È la funzione che calcola quanti millisecondi totali si risparmiano inserendo un video $V$ in una cache $C$.
* **Modalità Statica:** Se non riceve uno stato pregresso, confronta la latenza della cache solo rispetto al **Data Center**. 
* **Modalità Marginale:** Se riceve lo stato attuale delle latenze (`current_best_latency`), calcola il guadagno rispetto alla **miglior alternativa esistente**. 

**Pseudocodice:**
```python
1. TotalBenefit <- 0
2. Per ogni Endpoint E connesso alla Cache C:
3.     Richieste <- GetRequests(Video, Endpoint)
4.     If Richieste == 0: Continue
5.     
6.     # Determina il target da battere
7.     If (Modalità Marginale):
8.         Baseline <- CurrentBestLatency[(Video, Endpoint)]
9.     Else (Modalità Statica):
10.        Baseline <- DCLatency[Endpoint]
11.    
12.    # Calcola risparmio solo se migliorativo
13.    If (Latenza(Cache, E) < Baseline):
14.        Risparmio <- (Baseline - Latenza(Cache, E))
15.        TotalBenefit += Risparmio * Richieste
16.
17. Ratio <- TotalBenefit / Size(Video)
18. Return TotalBenefit, Ratio

In [3]:
from typing import Optional

def compute_benefit(video_id: int, cache_id: int, data: ProblemData, 
                    current_best_latency: Optional [Dict[Tuple[int, int], int]] = None) -> Tuple[float, float]:
    """
    INPUT:
        video_id: ID del video da valutare.
        cache_id: ID della cache in cui considerare il piazzamento.
        data: Struttura dati contenente le informazioni del problema.
        (Opzionale) current_best_latency: Dizionario (video_id, endpoint_id) -> miglior latenza attuale.
    OUTPUT:
        beneficio totale: Tempo totale risparmiato in millisecondi.
        beneficio statico per video: Tempo risparmiato normalizzato per dimensione video.
    """
    video_size = data.video_sizes[video_id]
    if video_size == 0: 
        return 0.0, 0.0
    
    total_benefit = 0
    
    # Iteriamo solo sugli endpoint connessi alla cache in input
    for endpoint_id in data.cache_to_endpoints[cache_id]:
        
        # 1. Controlliamo se questo endpoint vuole il video
        num_requests = data.requests_map.get((video_id, endpoint_id), 0)
        if num_requests == 0: 
            continue
        
        # 2. Recuperiamo le latenze della cache e del dc di questo endpoint
        cache_lat = data.cache_endpoint_latency[(cache_id, endpoint_id)]
        dc_latency = data.endpoints[endpoint_id]['datacenter_latency']

        # 3. Calcolo beneficio
        # In base al parametro confrontiamo rispetto a dc oppure rispetto alla best attuale
        if current_best_latency is not None:
            compare_lat = current_best_latency.get((video_id, endpoint_id), dc_latency)
        else:
            compare_lat = dc_latency
       
        if cache_lat < compare_lat:
            latency_saved = compare_lat - cache_lat
            total_benefit += num_requests * latency_saved
            
    return total_benefit, (total_benefit / video_size)

### Costruzione dell'heap: `build_initial_heap`
Questa funzione costruisce la coda di priorità iniziale tramite la funzione `compute_benefit`

Per efficienza, non testa tutte le combinazioni possibili ($V \times C$), ma solo i video che sono effettivamente richiesti dagli endpoint connessi a ciascuna cache

**Nota:** tale funzione è usata indifferentemente sia dall'algoritmo Statico che dal Marginale poichè in fase di costruzione dell'heap l'unica alternativa per tutti è il Data Center.

**Pseudocodice:**
```python
1. Heap <- Lista Vuota
2. Per ogni Cache C nel sistema:
3.     # Ottimizzazione: Considera solo video richiesti in zona
4.     Candidati <- GetVideosRequestedByEndpointsOf(C)
5.     
6.     Per ogni Video V in Candidati:
7.         If (Size(V) <= Capacity(C)):
8.             # Chiama la funzione di scoring 
9.             (Benefit, Ratio) <- compute_benefit(V, C)
10.            
11.            If Ratio > 0:
12.                # Inserisci in Heap (Negativo per Max-Heap)
13.                Heap.push(-Ratio, V, C, Benefit)
14.
15. Return Heap

In [4]:
import heapq
TQDM_UPDATE_INTERVAL = 1000     # Intervallo aggiornamento barre progresso

def build_initial_heap(data: ProblemData) -> List:
    """
    INPUT:
        data: Oggetto ProblemData con le strutture dati.
        
    OUTPUT:
        heap: Lista usata come Priority Queue con tuple (-ratio, video_id, cache_id).
    """
    heap = []

    # 1. Totale candidati
    total_candidates = sum(len(data.cache_to_requested_videos[c]) for c in range(data.C))

    # 2. Avviamo la barra
    # miniters aggiorna la grafica ogni tanto per non rallentare troppo
    with tqdm(total=total_candidates, desc="Building Heap", unit="cand", miniters=TQDM_UPDATE_INTERVAL) as pbar:
    
        # 3. cicla su tutte le cache e i video candidati
        for cache_id in range(data.C):
            # Prendiamo solo i video richiesti dagli endpoint di questa cache
            candidates = data.cache_to_requested_videos[cache_id]
            
            for video_id in candidates:
                # Check fisico di capacità
                if data.video_sizes[video_id] <= data.X:
                    # Calcolo beneficio statico (confronto con Data Center)
                    tot_ben, ratio = compute_benefit(video_id, cache_id, data)
                    
                    if ratio > 0:
                        # Heap di massimo (usiamo ratio negativo, di default è min-heap)
                        heapq.heappush(heap, (-ratio, video_id, cache_id, tot_ben))                        
            
            pbar.update(len(candidates))

    print(f"Heap populated with {len(heap):,} candidates.")                                
    return heap


### 🅰️ Strategia 1: Greedy Statico (con parametro $K$)
Questo approccio calcola il beneficio di un video in modo "assoluto", considerando solo il risparmio rispetto al Data Center.
* **Problema:** Tende a replicare lo stesso video su tutte le cache, sprecando capacità.
* **Soluzione Euristica:** Introduciamo un limite **$K$** (es. $K=2$ o $K=3$). Un video non può essere presente in più di $K$ cache in tutto il sistema.

**Pseudocodice:**
```python
1. Heap <- Calcola beneficio statico per ogni coppia (Video, Cache)
2. While Heap non è vuota:
3.     (Video, Cache) <- Estrai candidato con Ratio migliore
4.     
5.     If (Video entra in Cache) AND (CopieTotali(Video) < K):
6.         Assegna Video a Cache
7.         Aggiorna spazio occupato
8.         Incrementa contatore copie per Video

In [5]:
GREEDY_K = 3                     # Max copie per video in tutto il sistema
from utils import print_stats

def run_greedy(data: ProblemData, max_copies_per_video: int) -> Dict[int, Set[int]]:
    """
    INPUT:
        data: Oggetto ProblemData contenente tutte le strutture dati del problema 
        max_copies_per_video (int): Limite euristico 'K' per il numero massimo di copie 
                                    dello stesso video ammesse in tutta la rete.
    OUTPUT:
        cache_assignments: Dizionario cache_id -> set di video_id assegnati.
    """
    print(f"Starting Greedy (K={max_copies_per_video})")
    
    # 1. Setup Strutture
    cache_assignments = defaultdict(set)
    cache_used_space = np.zeros(data.C, dtype=int)
    video_copy_count = defaultdict(int) # video_id -> numero di copie assegnate

    # 2. Riempimento Heap
    heap = build_initial_heap(data)

    # Contatori
    metric_pop = 0

    # 3. Ciclo Greedy 
    with tqdm(total=len(heap), desc="Greedy Loop Exec", unit="step") as pbar:
        while heap:
            #ignoriamo il beneficio totale (_) nel greedy statico
            _, video_id, cache_id, _ = heapq.heappop(heap)
            metric_pop += 1
            pbar.update(1)

            # A. Vincolo Spazio
            if cache_used_space[cache_id] + data.video_sizes[video_id] > data.X:
                continue
                
            # B. Vincolo Duplicati Locali (è già qui?)
            if video_id in cache_assignments[cache_id]:
                continue
                
            # C. Vincolo Copie Globali (K) 
            if video_copy_count[video_id] >= max_copies_per_video:
                continue
                
            # Assegnazione a questa cache
            cache_assignments[cache_id].add(video_id)
            cache_used_space[cache_id] += data.video_sizes[video_id]
            video_copy_count[video_id] += 1
    
    # 4. Report
    print("Greedy Done.")
    print(f"[Heap stats] Total Pops: {metric_pop:,}")
    print_stats(cache_assignments, cache_used_space, data)
    
    return cache_assignments

### 🅱️ Strategia 2: Greedy Marginale
Questa strategia è più raffinata: invece di usare un limite artificiale $K$, calcola il beneficio in base allo **stato corrente** della rete.

* **Problema:** Il beneficio di un video non è costante. Se un endpoint è già servito da una cache molto veloce, aggiungere lo stesso video a una seconda cache leggermente più lenta ha un beneficio quasi nullo. Il Greedy Statico non "vede" questo calo di utilità poichè fa riferimento esclusivamente al beneficio calcolato nella costruzione dell'heap (in cui l'alternativa ad uno specifico piazzamento era solo il datacenter e NON una cache più vicina))

* **Soluzione Dinamica:** Ad ogni passo, il beneficio viene calcolato come:
  $$\Delta = (\text{LatenzaAttuale} - \text{LatenzaNuovaCache}) \times \text{Richieste}$$

Ricalcolare il beneficio di *tutti* i candidati nella Heap dopo ogni assegnazione sarebbe troppo lento. Usiamo quindi il seguente approccio:
1. Estraiamo il miglior candidato secondo la stima vecchia.
2. Ricalcoliamo il suo beneficio secondo la **situazione attuale** del sistema.
3. Se è ancora valido (cioè non è diminuito drasticamente), lo confermiamo. Altrimenti, lo rimettiamo in coda con il valore aggiornato.

**Funzione di Utility:** Dopo ogni assegnazione, chiamiamo `update_latencies_for_video_in_cache` che aggiorna il dizionario `current_best_latency` per tutti gli endpoint connessi alla cache che richiedono il video. Questa funzione mantiene traccia della miglior latenza disponibile per ogni coppia (video, endpoint).

**Pseudocodice:**
```python
1. Heap <- Calcola beneficio marginale iniziale (vs Data Center)
2. While Heap non è vuota:
3.     (Stima, Video, Cache) <- Estrai candidato con Ratio migliore
4.     
5.     # Ricalcolo il beneficio reale nello stato attuale
6.     RatioReale <- CalcolaBeneficio(Video, Cache, LatenzeAttuali)
7.     
8.     If (RatioReale < Stima):
9.         # Il beneficio è sceso (dato obsoleto): Reinserisco in Heap
10.        Heap.push(RatioReale, Video, Cache)
11.    Else If (Video entra in Cache):
12.        # Stima corretta e c'è spazio: Assegno!
13.        Assegna Video a Cache
14.        Aggiorna spazio occupato
15.        Aggiorna LatenzeAttuali (Nuova Baseline per calcoli futuri)
```

In [ ]:
FLOAT_TOLERANCE = 1e-6          # Tolleranza per confronto float nel calcolo dello score 
from utils import update_latencies_for_video_in_cache

def run_greedy_marginal(data: ProblemData) -> Dict[int, Set[int]]:
    """
    INPUT:
        data: Oggetto ProblemData contenente tutte le strutture dati del problema

    OUTPUT:
        cache_assignments: Dizionario cache_id -> set di video_id assegnati.

    """
    print("Starting Marginal Greedy")
    
    # 1. Setup Strutture
    cache_assignments = defaultdict(set)
    cache_used_space = np.zeros(data.C, dtype=int)
    
    # Inizializzazione dizionario (video_id, endpoint_id) -> miglior latenza attuale
    # All'inizio la best latency per tutti è il Data Center
    current_best_latency = {
        (v, e): data.endpoints[e]['datacenter_latency'] for (v, e) in data.requests_map.keys()
    }

    # 2. Riempimento Heap
    # Anche se abbiamo current_best_latency, all'inizio è uguale al Data Center,
    # quindi il beneficio calcolato è identico al greedy statico
    heap = build_initial_heap(data)

    # Contatori
    metric_reinsertions = 0
    metric_pop = 0

    # 3. Ciclo Greedy Marginal
    with tqdm(total=len(heap), desc="Marginal Greedy Loop Exec", unit="step") as pbar:
        while heap:

            #Estrae il candidato con il beneficio stimato più alto
            neg_ratio, video_id, cache_id, _ = heapq.heappop(heap)
            metric_pop += 1
            pbar.update(1)

            # A. Vincolo Duplicati Locali (è già qui?)
            if video_id in cache_assignments[cache_id]:
                continue
            
            # Ricalcolo beneficio con lo stato attuale delle latenze migliori
            # ho estratto il candidato migliore, ma il beneficio potrebbe essere cambiato dopo alcune assegnazioni
            real_ben, real_ratio = compute_benefit(video_id, cache_id, data, current_best_latency)
            
            # B. Se il beneficio è sceso a zero, scartiamo
            if real_ratio <= 0: 
                continue
            
            # C. Se il beneficio è cambiato significativamente rispetto alla stima nella heap
            #    non scartiamo l'opzione, ma rimettiamo nella heap col nuovo valore corretto
            #    (Usiamo una tolleranza relativa per i float)
            if abs(real_ratio - (-neg_ratio)) > FLOAT_TOLERANCE:
                heapq.heappush(heap, (-real_ratio, video_id, cache_id, real_ben))
                metric_reinsertions += 1
                pbar.total += 1
                pbar.refresh()
                continue
            
            # D. Vincolo Spazio
            # Questo controllo deve essere fatto dopo il ricalcolo del beneficio reale
            # la cache potrebbe essersi riempita mentre l'elemento era nella heap
            if cache_used_space[cache_id] + data.video_sizes[video_id] > data.X:
                continue
                
            # Assegnazione a questa cache
            cache_assignments[cache_id].add(video_id)
            cache_used_space[cache_id] += data.video_sizes[video_id]
            
            # Aggiorniamo la mappa delle migliori latenze per gli endpoint connessi a questa cache
            update_latencies_for_video_in_cache(video_id, cache_id, data, current_best_latency)


    # 4. Report finale
    print("Greedy Marginal Done.")
    print(f"[Heap stats] Total Pops: {metric_pop:,} | Total Re-insertions: {metric_reinsertions:,}")
    print_stats(cache_assignments, cache_used_space, data)
    return cache_assignments

### Calcolo soluzione e score iniziale


Generiamo la soluzione iniziale (Greedy) e calcoliamo la **mappa delle latenze attuali**.
Questo "snapshot" dello stato serve a valutare lo score iniziale ed è il prerequisito essenziale per la **Local Search**.

La funzione `calculate_best_latencies` ha un duplice compito: costruire la mappa completa delle latenze ottimali e calcolare lo score della soluzione corrente.

Partendo da una configurazione delle cache (la soluzione Greedy), la funzione inizializza per ogni coppia (video, endpoint) presente nelle richieste la latenza di riferimento al Data Center, rappresentando il caso peggiore. 

Successivamente, usando tale configurazione itera su tutte le cache popolate e, per ogni video assegnato, chiama `update_latencies_for_video_in_cache` che calcola per ogni endpoint connesso a quella cache (e che richiede un video presente nella cache) il risparmio totale in millisecondi.

tali valori vengono accumulati durante il processo ottenendo il **tempo totale risparmiato**

In [8]:
def calculate_best_latencies(data: ProblemData, assignments: Dict[int, Set[int]]) -> Tuple[Dict, float]:
    """
    INPUT:
        data: Oggetto ProblemData con le strutture del problema.
        assignments: Dizionario cache_id -> set di video assegnati (soluzione greedy).
    OUTPUT:
        best_latency: Dizionario (video_id, endpoint_id) -> latenza minima attuale.
        total_saved: Float rappresentante la somma totale dei millisecondi risparmiati
                     rispetto al Data Center con la configurazione attuale.
    """
    best_latency = {}
    
    # 1. Base: Datacenter
    for (v, e) in data.requests_map.keys():
        best_latency[(v, e)] = data.endpoints[e]['datacenter_latency']
        
    # 2. Miglioramento con cache
    total_saved = 0.0
    for c_id, videos in assignments.items():
        for v_id in videos:
            total_saved += update_latencies_for_video_in_cache(v_id, c_id, data, best_latency)

    return best_latency, total_saved

con un flag indichiamo la soluzione da usare

In [9]:
USE_MARGINAL_GREEDY = True
SCORE_MULTIPLIER = 1000

# Imposta seed per riproducibilità
print("Setting random seed to 42 for reproducibility")
np.random.seed(42)
print("Random seed set.\n")

if USE_MARGINAL_GREEDY:
    greedy_solution = run_greedy_marginal(data)
else:
    greedy_solution = run_greedy(data, max_copies_per_video=GREEDY_K)


initial_best_latency, initial_total_saved = calculate_best_latencies(data, greedy_solution)
initial_score = int((initial_total_saved / data.total_requests) * SCORE_MULTIPLIER)
print(f"Initial Score (Greedy): {initial_score:,} points")

Setting random seed to 42 for reproducibility
Random seed set.

Starting Marginal Greedy


Building Heap: 100%|██████████| 183/183 [00:00<00:00, 182534.51cand/s]


Heap populated with 183 candidates.


Marginal Greedy Loop Exec: 100%|██████████| 250/250 [00:00<00:00, 2688.45step/s]

Greedy Marginal Done.
[Heap stats] Total Pops: 250 | Total Re-insertions: 67
Result: 29/100 unique videos cached, 51 total placements, 93.6% space utilization
Initial Score (Greedy): 507,906 points


## 5. Local Search


La Local Search esplora l'intorno della soluzione attuale effettuado operazioni di scambio che aumentano lo score totale. 

### Operazione di Swap
Iteriamo su ogni cache e tentiamo di sostituire un video presente in una cache ($V_{out}$) con uno presente nel datacenter ($V_{in}$).
Lo scambio viene confermato se e solo se:
1.  **Validità:** Il nuovo video entra fisicamente nella cache (rispetto della capacità $X$).
2.  **Miglioramento:** Il punteggio totale aumenta ($\Delta Score > 0$).

### Calcolo del Delta ($\Delta$)
La funzione `calculate_swap_delta` risponde alla domanda: *"Conviene scambiare il video $V_{out}$ con $V_{in}$ nella cache $C$?"*
Ricalcolare lo Score totale ($R$ richieste) ad ogni tentativo sarebbe troppo oneroso. Calcoliamo quindi, senza modificare lo stato attuale, la differenza di score solo sui video interessati allo scambio

$$\Delta = \text{Gain}(V_{in}) - \text{Loss}(V_{out})$$

* **Gain:** Calcolato usando la logica marginale (quanto migliora la latenza per chi ora trova $V_{in}$ nella cache).
* **Loss:** Se rimuoviamo $V_{out}$, gli utenti connessi non tornano necessariamente al Data Center. La funzione cerca la **Second Best Latency**: esiste un'altra cache vicina che ha lo stesso video? Se sì, la perdita è minima.

**Pseudocodice:**
```python
1. Gain <- ComputeBenefit(VideoIn, Cache, BestLatency)
2. If Gain <= 0: 
3.     Return -1 (Il nuovo video non porta benefici, inutile proseguire)
4. 
5. Loss <- 0
6. SecondBestMap <- Mappa Vuota (serve per l'aggiornamento successivo)
7. 
8. Per ogni Endpoint E connesso alla Cache:
9.     If E non richiede VideoOut: Continue
10.    
11.    # Verifichiamo se la cache corrente era essenziale per questo endpoint
12.    If Latenza(Cache, E) == BestLatency(VideoOut, E):
13.        # Era la sorgente migliore! Dobbiamo trovare l'alternativa
14.        NewBestLat <- FindSecondBestLatency(VideoOut, E, Cache, ...)
15.        SecondBestMap[E] <- NewBestLat
16.        
17.        # La perdita è il peggioramento di latenza pesato per le richieste
18.        DeltaPerdita <- (NewBestLat - BestLatency(VideoOut, E)) * Richieste
19.        Loss += DeltaPerdita
20.
21. Delta <- Gain - Loss
22. Return Delta, SecondBestMap

In [10]:
from utils import find_second_best_latency

def calculate_swap_delta(video_out: int, video_in: int, cache_id: int, 
                         data: ProblemData, best_latency: Dict, assignments: Dict) -> Tuple[float, Dict[int, int]]:
    """
    INPUT:
        video_out: ID del video candidato alla rimozione.
        video_in: ID del video candidato all'inserimento.
        cache_id: ID della cache su cui effettuare lo scambio 
        data: Oggetto ProblemData.
        best_latency: Dizionario (video_id, endpoint_id) -> latenza minima attuale.
        assignments: Soluzione iniziale (dizionario cache -> set di video_id assegnati).
    OUTPUT:
        delta: Float che rappresenta la variazione di tempo risparmiato.
               - Valore > 0: Lo scambio migliora la soluzione.
               - Valore <= 0: Lo scambio è peggiorativo o neutro.
        second_best_map: Dizionario endpoint_id -> nuova miglior latenza per video_out.
                        Serve per evitare di ricalcolare in update_state_after_swap.
    """
    # Inizializza gain, il risparmio ottenuto aggiungendo video_in
    gain, _ = compute_benefit(video_in, cache_id, data, current_best_latency=best_latency)
    if gain <= 0:
        return -1.0, {}

    
    # Calcolo perdita (Rimuovendo video_out)
    loss = 0
    second_best_map = {}  # endpoint_id -> nuova best latency per video_out
    
    for e in data.cache_to_endpoints[cache_id]:

        # Se non c'è richiesta per video_out da tutti gli endpoint connessi, non c'è perdita
        if (video_out, e) not in data.requests_map:
            continue
        
        # Se questa cache non era la migliore, rimuovere il video_out non fa danni 
        current_lat = data.cache_endpoint_latency[(cache_id, e)]
        if current_lat != best_latency[(video_out, e)]:
            continue

        # Se era la migliore, dobbiamo trovare chi prende il suo posto
        new_best = find_second_best_latency(video_out, e, cache_id, data, assignments)
        second_best_map[e] = new_best  # Salviamo per riusarlo dopo
            
        loss += (new_best - best_latency[(video_out, e)]) * data.requests_map[(video_out, e)]
    
    delta = gain - loss
    return delta, second_best_map

### Aggiornamento di Stato
Se $\Delta > 0$, la funzione `update_state_after_swap` applica fisicamente lo scambio e aggiorna la mappa globale `best_latency` per
rendere permanente la modifica.


In [ ]:
def update_state_after_swap(video_out: int, video_in: int, cache_id: int, 
                            data: ProblemData, best_latency: Dict, assignments: Dict,
                            second_best_map: Dict[int, int]):
    """
    INPUT:
        video_out: ID del video da rimuovere.
        video_in: ID del video da aggiungere.
        cache_id: ID della cache modificata.
        data: Oggetto ProblemData.
        best_latency: Mappa da aggiornare.
        assignments: Dizionario assegnamenti da aggiornare.
        second_best_map: Dizionario endpoint_id -> nuova best latency per video_out,
                        precalcolato da calculate_swap_delta per evitare ricalcoli.
    OUTPUT:
        Nessuno (Le strutture dati sono modificate per riferimento).
    """
    # 1. Modifica Assegnamento
    assignments[cache_id].remove(video_out)
    assignments[cache_id].add(video_in)
    
    # 2. Aggiorna Best Latency per video_in 
    update_latencies_for_video_in_cache(video_in, cache_id, data, best_latency)

    # 3. Aggiorna Best Latency per video_out usando i valori precalcolati
    for e, new_best in second_best_map.items():
        key = (video_out, e)
        best_latency[key] = new_best

### Spiegazione e implementazione della funzione local search

La funzione `run_local_search` gestisce il processo di ottimizzazione. Ad ogni iterazione vengono analizzate tutte le cache cercando un possibile swap.

Per evitare di rimanere bloccati in ottimi locali, l'algoritmo non è deterministico ma **stocastico**:
1.  **Random Shuffle:** L'ordine delle cache analizzate viene rimescolato a ogni "Pass".
2.  **Campionamento:** Invece di testare *tutte* le combinazioni, selezioniamo un sottoinsieme casuale di candidati IN e OUT ad ogni iterazione.
3.  **First Improvement:** Appena viene individuato uno swap valido ($\Delta > 0$), viene applicato immediatamente e si passa alla cache successiva.

L'algoritmo termina appena vengono effettuate 5 iterazioni senza trovare swap disponibili oppure quando vengono effettuati 5000 swap consecutivi

N.B. le print di log della funzione vengono salvate in un apposito file di log

**Pseudocodice:**
```python
1. Inizializza ScoreCorrente e Contatori (PassiSenzaMiglioramento = 0)
2. While (PassiSenzaMiglioramento < MaxPassi) AND (SwapConsecutivi < MaxSwaps):
3.     CacheOrder <- PermutazioneCasuale(TutteLeCache)
4.     SwapTrovatoInQuestoPass <- False
5.     
6.     For Cache in CacheOrder:
7.         # Selezione Stocastica: Prendo solo un campione di video da testare
8.         CandidatiOUT, CandidatiIN <- EstraiSottoinsiemiCasuali(Cache)
9.         
10.        For VideoOut in CandidatiOUT:
11.            For VideoIn in CandidatiIN:
12.                
13.                # 1. Verifica Vincoli (Capacità Cache)
14.                If NOT IsSwapValid(VideoOut, VideoIn, ...): Continue
15.                
16.                # 2. Calcolo Delta Score
17.                Delta <- CalculateSwapDelta(VideoOut, VideoIn, ...)
18.                
19.                If Delta > 0:
20.                    # Miglioramento trovato: Applica subito (First Improvement)
21.                    UpdateStateAfterSwap(VideoOut, VideoIn, ...)
22.                    ScoreCorrente += Delta
23.                    SwapTrovatoInQuestoPass <- True
24.                    Break (Esce dai cicli candidati)
25.            
26.            If SwapTrovatoInQuestoPass: Break (Passa alla prossima Cache)
27.            
28.    If SwapTrovatoInQuestoPass:
29.        PassiSenzaMiglioramento <- 0
30.    Else:
31.        PassiSenzaMiglioramento += 1
32.        Reset ContatoreSwapConsecutivi

In [12]:
MAX_PASSES_WITHOUT_IMPROVEMENT = 5  # Stop dopo N pass consecutivi senza miglioramenti
MAX_VIDEO_OUT_PER_CACHE = 10   # Max video da provare a rimuovere per cache 
MAX_VIDEO_IN_PER_SWAP = 20     # Max video candidati per swap
MAX_SUCCESSFUL_SWAPS = 5000    # Stop dopo N swap consecutivi andati a buon fine 

from utils import is_swap_valid, get_stochastic_candidates

def run_local_search(assignments: Dict[int, Set[int]],
                     data: ProblemData,
                     max_passes_no_improvement: int,
                     best_latency: Dict,        
                     current_total_saved: float,
                     max_video_out_per_cache: int,
                     max_video_in_per_swap: int,
                     max_successful_swaps: int,
                     log_filename: str = "local_search_log.txt"
                     ) -> tuple:
    """
    INPUT:
        assignments: Soluzione iniziale (dizionario cache -> set di video_id assegnati).
        data: Oggetto ProblemData.
        max_iter: Numero massimo di iterazioni senza miglioramenti globali o totali.
        best_latency: Dizionario (video_id, endpoint_id) -> latenza minima attuale.
        current_total_saved: Risparmio totale iniziale in millisecondi.
        max_video_out_per_cache: Max numero di video da provare a rimuovere per cache (default 10).
        max_video_in_per_swap: Max numero di video candidati da provare per ogni rimozione (default 20).
        max_successful_swaps: Max numero di swap consecutivi andati a buon fine prima di fermarsi (default 5000).
        log_filename: Nome del file di log per salvare i dettagli dell'esecuzione.
    OUTPUT:
        assignments: La soluzione migliorata (dizionario cache -> set di video_id assegnati)
        current_score: Il punteggio finale raggiunto.
    """

    print(f"Local Search started") 
    with open(log_filename, 'w') as f_log:

        f_log.write("Starting Local Search...\n")
        f_log.flush() # Forza la scrittura su disco immediata

        print(f"Local Search started. Log info in: {log_filename}")

        total_saved_acc = current_total_saved
        current_score = int((total_saved_acc / data.total_requests) * SCORE_MULTIPLIER)   

        #counters
        total_swaps = 0
        consecutive_successful_swaps = 0  # Contatore swap consecutivi andati a buon fine
        pass_count = 0 # Iterazione completa su tutte le cache
        passes_without_improvement = 0 # Contatore pass senza miglioramenti
        last_swap_info = None  # Memorizza info ultimo swap per stamparlo a fine pass

        while passes_without_improvement < max_passes_no_improvement and consecutive_successful_swaps < max_successful_swaps:
            pass_count += 1
            swap_found_this_pass = False

            # Randomizzo l'ordine delle cache per non favorire sempre le stesse
            cache_order = np.random.permutation(data.C)

            iterator = tqdm(cache_order, 
                            desc=f"Pass {pass_count}", 
                            leave=False, # leave=False fa scomparire la barra quando finisce/resetta,
                            unit="cache")

            "CICLO SU TUTTE LE CACHE"
            #iteriamo su tutte le cache in ordine random
            for cache_id in iterator:
                if swap_found_this_pass:  # Se abbiamo già trovato uno swap, esci
                    break

                "RECUPERO CANDIDATI SWAP"
                # Selezione stocastica di candidati OUT e IN
                videos_in_cache, candidates, used_space = get_stochastic_candidates(
                    cache_id, assignments, data, max_video_out_per_cache, max_video_in_per_swap
                )
                
                # Se la cache è vuota o non ci sono candidati, passa alla prossima
                if not videos_in_cache or not candidates:
                    continue

                "CICLO SU TUTTI I POSSIBILI SWAP"
                for video_out in videos_in_cache:
                    if swap_found_this_pass:  # Se abbiamo già trovato uno swap, esci
                        break
                    
                    for video_in in candidates:
                        # Verifica validità dello swap
                        if not is_swap_valid(video_out, video_in, cache_id, used_space, data, assignments):
                            continue

                        # Calcolo Delta + eventuale seconda best latency map
                        delta, second_best_map = calculate_swap_delta(video_out, video_in, cache_id, data, best_latency, assignments)

                        # Applicazione Swap
                        if delta > 0:
                            update_state_after_swap(video_out, video_in, cache_id, data, best_latency, assignments, second_best_map)
                            
                            total_saved_acc += delta
                            current_score = int((total_saved_acc / data.total_requests) * SCORE_MULTIPLIER)
                            swap_found_this_pass = True
                            # Salvo info per stamparla a fine pass
                            last_swap_info = (video_out, video_in, cache_id, current_score)
                            iterator.set_postfix({'Score': f"{current_score:,}"})
                            break # Esce dal ciclo video_in

            # Se abbiamo fatto uno swap, chiudiamo l'iterator e ricominciamo dal while 
            if swap_found_this_pass:
                passes_without_improvement = 0 
                total_swaps += 1
                consecutive_successful_swaps += 1

                iterator.close()
                video_out, video_in, cache_id, score = last_swap_info
                f_log.write(f"Pass {pass_count} completed. Swap [{video_out} out] -> [{video_in} in] | (Cache {cache_id}), Score: {score:,} | Consecutive swaps: {consecutive_successful_swaps}\n")
                f_log.flush()
                continue  # pass ad una nuova iterazione del while

            # Se sono qui vuol dire che nessun swap è stato trovato in questo pass
            passes_without_improvement += 1
            consecutive_successful_swaps = 0  

            iterator.close()
            f_log.write(f"Pass {pass_count} completed. No improvement. Consecutive passes without improvement: {passes_without_improvement}/{max_passes_no_improvement}. Score: {current_score:,}\n")
            f_log.flush()
        
        # Determina motivo terminazione
        if consecutive_successful_swaps >= max_successful_swaps:
            f_log.write(f"\nLocal Search terminated: reached max consecutive successful swaps ({consecutive_successful_swaps}/{max_successful_swaps})\n")
        else:
            f_log.write(f"\nLocal Search terminated: no improvement for {max_passes_no_improvement} consecutive passes\n")
        f_log.write(f"Total: {pass_count} passes, {total_swaps} total swaps.\n")

    print(f"Local Search terminated")  
    return assignments, current_score


## Score finale e salvataggio risultato

Chiamiamo la funzione principale. Il risultato finale è scritto su file tramite la funzione di utility write_output. I dati vengono salvati nel seguente modo come richiesto dalla traccia.

### Esempio
<div align="center">
    <img src="immagini/esempio_output.png" width="70%">
</div>

In [13]:
from utils import write_output

OUTPUT_FILE = r"output_prova_notebook.txt"
LOG_FILE = r"debug_local_search.txt"

final_solution, final_score = run_local_search( greedy_solution, 
                                                data, 
                                                max_passes_no_improvement=MAX_PASSES_WITHOUT_IMPROVEMENT,
                                                best_latency=initial_best_latency,      
                                                current_total_saved=initial_total_saved,
                                                max_video_out_per_cache=MAX_VIDEO_OUT_PER_CACHE,
                                                max_video_in_per_swap=MAX_VIDEO_IN_PER_SWAP,
                                                max_successful_swaps=MAX_SUCCESSFUL_SWAPS,
                                                log_filename=LOG_FILE
                                                )

print(f"Final Score: {final_score:,} points")
improvement = final_score - initial_score
print(f"Local search improvement: +{improvement:,} points")
write_output(final_solution, OUTPUT_FILE)

Local Search started
Local Search started. Log info in: debug_local_search.txt


Local Search terminated
Final Score: 507,906 points
Local search improvement: +0 points
Writing output to output_prova_notebook.txt...
Write complete.


## 6. Analisi dei Risultati

### Classifica finale


| Strategia | me_at_the_zoo | trending_today | videos_worth | kittens | **TOTALE** | Qualità Greedy | Dipendenza da LS | Scalabilità |
|-----------|---------------|----------------|--------------|---------|------------|----------------|------------------|-------------|
| **Static K=1** | 475,634 | 499,966 | 475,147 | 1,012,920 | **2,463,667** (93.2%) | Media-Alta | Media | Eccellente |
| **Static K=2** | 495,799 | 432,597 | 425,071 | 866,121 | **2,219,588** (84.0%) | Bassa-Media | Molto Alta | Problematica |
| **Static K=3** | 483,732 | 380,119 | 430,858 | 744,968 | **2,039,677** (77.2%) | Bassa | Molto Alta | Problematica |
| **Marginal** | 507,906 | 499,966 | 608,946 | 1,021,709 | **2,638,527** ⭐ (100%) | Alta | Minima | Eccellente |

**Confronto con classifica Hash Code 2017:**
- Vincitore mondiale: 2,651,999 punti
- Soluzione migiore: **2,638,527** punti
- Gap: **-13,472** punti (-0.5%)

<div align="center">
    <img src="immagini/grafico_3_ls_improvement.png" width="65%">
</div>

<div align="center">
    <img src="immagini/grafico_2_tempo_esecuzione.png" width="55%">
    <img src="immagini/grafico_6_totali.png" width="44%">
</div>


-  **Reinserimenti**: Su kittens registra 89% re-insertions.

-  **Space utilization**: Marginal raggiunge 100% utilizzo su 2 dataset (videos_worth_spreading, kittens)

-  **K>1 controproducente**: La sovra-replicazione satura le capacità delle cache senza diversificare l'offerta, penalizzando video meno popolari ma strategicamente importanti per endpoint specifici

-  **trending_today - Pattern video popolari**: Dataset dominato da una manciata di video "trending". Sovra-replicarli (K=2/K=3) non migliora lo score perché il problema non è la disponibilità ma la distribuzione geografica

-  **videos_worth_spreading - Caso critico**: Tutte le strategie statiche raggiungono il limite 5,000 swaps senza convergere (potrebbero migliorare ulteriormente)

-  **Scalabilità**: Dataset grande dimostra eccellente scalabilità di Marginal e K=1 su istanze complesse. K=2/K=3 raggiungono limite 5,000 swaps.

### Risultati dettagliati per dataset

#### DATASET 1: me_at_the_zoo.in


- 100 video, 10 endpoint, 100 requests, 10 caches
- Cache capacity: 100MB

| Strategia | Heap Candidates | Heap Stats | Videos Cached | Placements | Space Util | Greedy Score | Final Score | LS Improvement | LS Termination | Passes/Swaps | Time |
|-----------|----------------|------------|---------------|------------|------------|--------------|-------------|----------------|----------------|--------------|------|
| **K=1** | 183 | Pops: 183 | 32/100 | 32 | 76.2% | 402,352 | **475,634** | +73,282 (+18.2%) | 5 passes no impr. | 37/32 | 0.07s |
| **K=2** | 183 | Pops: 183 | 27/100 | 47 | 93.3% | 447,479 | **495,799** | +48,320 (+10.8%) | 5 passes no impr. | 42/37 | 0.07s |
| **K=3** | 183 | Pops: 183 | 25/100 | 56 | 94.1% | 427,839 | **483,732** | +55,893 (+13.1%) | 5 passes no impr. | 34/29 | 0.06s |
| **Marginal** | 183 | Pops: 250, Re-ins: 67 | 29/100 | 51 | 93.6% | 507,906 | **507,906** ⭐ | 0 | 5 passes no impr. | 5/0 | 0.11s |


#### DATASET 2: trending_today.in

- 10,000 video, 100 endpoint, 100,000 requests, 100 caches
- Cache capacity: 50,000MB

| Strategia | Heap Candidates | Heap Stats | Videos Cached | Placements | Space Util | Greedy Score | Final Score | LS Improvement | LS Termination | Passes/Swaps | Time |
|-----------|----------------|------------|---------------|------------|------------|--------------|-------------|----------------|----------------|--------------|------|
| **K=1** | 1,000,000 | Pops: 1,000,000 | 9993/10000 | 9993 | 99.9% | 499,966 | **499,966** ⭐ | 0 | 5 passes no impr. | 5/0 | 22.4s |
| **K=2** | 1,000,000 | Pops: 1,000,000 | 6893/10000 | 13786 | 99.9% | 378,343 | **432,597** | +54,254 (+14.3%) | 5 passes no impr. | 2594/2490 | 94.9s |
| **K=3** | 1,000,000 | Pops: 1,000,000 | 5527/10000 | 16385 | 99.9% | 310,654 | **380,119** | +69,465 (+22.4%) | 5 passes no impr. | 2563/2520 | 35.6s |
| **Marginal** | 1,000,000 | Pops: 1,000,000, Re-ins: 0 | 9993/10000 | 9993 | 99.9% | 499,966 | **499,966** ⭐ | 0 | 5 passes no impr. | 5/0 | 22.6s |


#### DATASET 3: videos_worth_spreading.in

- 10,000 video, 100 endpoint, 100,000 requests, 100 caches
- Cache capacity: 10,000MB

| Strategia | Heap Candidates | Heap Stats | Videos Cached | Placements | Space Util | Greedy Score | Final Score | LS Improvement | LS Termination | Passes/Swaps | Time |
|-----------|----------------|------------|---------------|------------|------------|--------------|-------------|----------------|----------------|--------------|------|
| **K=1** | 135,023 | Pops: 135,023 | 4274/10000 | 4274 | 99.7% | 176,888 | **475,147** | +298,259 (+168.6%) | Max 5000 swaps | 5000/5000 | 5.4s |
| **K=2** | 135,023 | Pops: 135,023 | 3135/10000 | 5619 | 99.9% | 224,801 | **425,071** | +200,270 (+89.1%) | Max 5000 swaps | 5000/5000 | 5.0s |
| **K=3** | 135,023 | Pops: 135,023 | 2666/10000 | 6544 | 99.9% | 266,700 | **430,858** | +164,158 (+61.5%) | Max 5000 swaps | 5000/5000 | 5.1s |
| **Marginal** | 135,023 | Pops: 170,501, Re-ins: 35,478 | 2279/10000 | 6110 | 100.0% | 608,277 | **608,946** ⭐ | +669 (+0.1%) | 5 passes no impr. | 447/342 | 20.8s |


#### DATASET 4: kittens.in.txt

- 10,000 video, 1,000 endpoint, 200,000 requests, 500 caches
- Cache capacity: 6,000MB

| Strategia | Heap Candidates | Heap Stats | Videos Cached | Placements | Space Util | Greedy Score | Final Score | LS Improvement | LS Termination | Passes/Swaps | Time |
|-----------|----------------|------------|---------------|------------|------------|--------------|-------------|----------------|----------------|--------------|------|
| **K=1** | 4,999,996 | Pops: 4,999,996 | 7475/10000 | 7475 | 97.6% | 1,008,497 | **1,012,920** | +4,423 (+0.4%) | 5 passes no impr. | 721/628 | 3.75h |
| **K=2** | 4,999,996 | Pops: 4,999,996 | 5331/10000 | 10503 | 98.4% | 777,220 | **866,121** | +88,901 (+11.4%) | Max 5000 swaps | 5000/5000 | 22min |
| **K=3** | 4,999,996 | Pops: 4,999,996 | 4395/10000 | 12887 | 98.7% | 657,000 | **744,968** | +87,968 (+13.4%) | Max 5000 swaps | 5000/5000 | 10min |
| **Marginal** | 4,999,996 | Pops: 9,487,229, Re-ins: 4,487,233 | 7407/10000 | 10815 | 100.0% | 1,021,680 | **1,021,709** ⭐ | +29 (~0%) | 5 passes no impr. | 507/323 | 2.9h |


In [14]:
def calculate_theoretical_upper_bound(data: ProblemData) -> int:
    """
    Calcola l'Upper Bound rilassando il vincolo di capacità delle cache.
    Ipotizza che ogni video richiesto possa essere servito dalla cache 
    più veloce connessa all'endpoint, indipendentemente dallo spazio.
    """
    max_possible_savings = 0.0
    
    # Iteriamo su tutte le richieste
    for (video_id, endpoint_id), num_requests in data.requests_map.items():
        
        dc_latency = data.endpoints[endpoint_id]['datacenter_latency']
        
        # Trova la latenza minima possibile tra tutte le connessioni fisiche (o DC)
        # Se l'endpoint non ha cache, min_latency sarà uguale a dc_latency
        min_latency = dc_latency
        
        # Controlliamo tutte le cache connesse a questo endpoint
        for cache_id, latency in data.endpoints[endpoint_id]['cache_connections']:
            if latency < min_latency:
                min_latency = latency
        
        # Risparmio ideale
        savings = (dc_latency - min_latency) * num_requests
        max_possible_savings += savings
        
    return int((max_possible_savings / data.total_requests) * 1000)

# Esempio di utilizzo
ub_score = calculate_theoretical_upper_bound(data)
print(f"📈 Theoretical Upper Bound (Infinite Capacity): {ub_score:,}")

📈 Theoretical Upper Bound (Infinite Capacity): 561,356


In [15]:
# --- ANALISI OTTIMALITÀ ---
ub_score = calculate_theoretical_upper_bound(data)
my_score = final_score

gap = ((ub_score - my_score) / ub_score) * 100
efficiency = 100 - gap

print("-" * 40)
print(f"🏆 PERFORMANCE ANALYSIS")
print(f"Upper Bound (Ideal):   {ub_score:,}")
print(f"Our Solution (Actual): {my_score:,}")
print(f"Optimality Gap:        {gap:.2f}%")
print(f"System Efficiency:     {efficiency:.2f}%")
print("-" * 40)

----------------------------------------
🏆 PERFORMANCE ANALYSIS
Upper Bound (Ideal):   561,356
Our Solution (Actual): 507,906
Optimality Gap:        9.52%
System Efficiency:     90.48%
----------------------------------------
